## Imports

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

if not hasattr(np, 'int'):
    np.int = int

seed = 7

## Dataset

In [ ]:
all_times_df = pd.read_csv('predictors_plus_target_calibrated.csv')
all_times_df = all_times_df.drop(['p6177_i0', 'p90004', 'p4079_i0_a0', 'p94_i0_a0', 'p4080_i0_a0', 'p93_i0_a0', 'p20002_i0_a0', 'p6177_i0', 'p30760_i0', 'p30870_i0'], axis=1)
all_times_df = all_times_df.dropna()
print('Number of Individuals ', len(all_times_df))

In [ ]:
label_encoder = LabelEncoder()
all_times_df['activity_class'] = label_encoder.fit_transform(all_times_df['activity_class'])

all_times_df['p31'] = pd.factorize(all_times_df['p31'])[0]
all_times_df['p2306_i0'] = pd.factorize(all_times_df['p2306_i0'])[0]
all_times_df['p2443_i0'] = pd.factorize(all_times_df['p2443_i0'])[0]

In [ ]:
X = all_times_df[['Mean', 'STD', 'skewR', 'kurtR', 'perc95', 'perc5', 'dPerc', 'Peak power', 'PPFd', 'Entropy', 'PF_sum', 'P_sum', 'p31','Steps']]
y = all_times_df['activity_class']

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

## Basic Approach

In [ ]:
rf_classifier = RandomForestClassifier(random_state=seed)
rf_classifier.fit(X_train_full, y_train_full)
y_pred = rf_classifier.predict(X_test)

In [ ]:
print("Model Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
for score in rf_classifier.feature_importances_:
    print(score)

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=rf_classifier.classes_)

print("Confusion Matrix:")
print(cm)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## GET RECOMMENDER FILE

In [ ]:
whole_file = pd.read_csv('whole_file_for_recommender.csv')
whole_file = whole_file[['filename', 'p21022','Mean', 'STD', 'skewR', 'kurtR', 'perc95', 'perc5', 'dPerc', 'Peak power', 'PPFd', 'Entropy', 'PF_sum', 'P_sum', 'p31','Steps']]
whole_file

In [ ]:
whole_file = whole_file.dropna()
print('Number of Individuals ', len(whole_file))

In [ ]:
whole_file['p31'] = pd.factorize(whole_file['p31'])[0]

In [ ]:
y_all = rf_classifier.predict(whole_file.iloc[:,2:])
len(y_all)

In [ ]:
whole_file['Predictions'] = y_all
whole_file = whole_file[['filename', 'p21022', 'Predictions']]
whole_file.to_csv('recommender_file_met.csv')

## Scaled

In [ ]:
scaler = StandardScaler()
X_train_full_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

In [ ]:
rf_classifier = RandomForestClassifier(random_state=seed)
rf_classifier.fit(X_train_full_scaled, y_train_full)
y_pred = rf_classifier.predict(X_test_scaled)

In [ ]:
print("Model Accuracy:", accuracy_score(y_test, y_pred))

## Skewness

In [ ]:
from scipy.stats import skew

X_copy = X.copy(deep=True)

numeric_feats = X_copy.select_dtypes(include=['int64', 'float64'])
skewed_feats = numeric_feats.apply(lambda x: skew(x.dropna())).sort_values(ascending=False)

high_skew = skewed_feats[abs(skewed_feats) > 0.75]
print(high_skew)


In [ ]:
plt.subplot(1, 2, 1)
sns.histplot(X['P_sum'].dropna(), kde=True, color='skyblue')
plt.title(f'Before Log Transform')

plt.subplot(1, 2, 2)
sns.histplot(np.log1p(X['P_sum'].dropna()), kde=True, color='salmon')
plt.title(f'After Log Transform')

plt.tight_layout()
plt.show()

In [ ]:
skewed_features = high_skew.index
X_copy[skewed_features] = np.log1p(X_copy[skewed_features])

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X_copy, y, test_size=0.2, random_state=seed)

In [ ]:
scaler = StandardScaler()
X_train_full_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')


X_train_full_imputed = imputer.fit_transform(X_train_full_scaled)
X_test_imputed = imputer.transform(X_test_scaled)

In [ ]:
# Now train the model
rf_classifier = RandomForestClassifier(random_state=seed)
rf_classifier.fit(X_train_full_imputed, y_train_full)
y_pred = rf_classifier.predict(X_test_imputed)

In [ ]:
print("Model Accuracy:", accuracy_score(y_test, y_pred))

## Bayesian

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

In [ ]:
scaler = StandardScaler()
X_train_full_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from skopt.space import Integer, Categorical

param_space = {
    'n_estimators': Integer(100, 2000),
    'max_depth': Integer(2, 50),
    'min_samples_split': Integer(2, 50),
    'min_samples_leaf': Integer(1, 50),
    'max_features': Categorical(['sqrt', 'log2', None, 0.2, 0.5, 0.8])
}


In [ ]:
from skopt import BayesSearchCV

opt = BayesSearchCV(
    estimator=RandomForestClassifier(random_state=42, criterion='gini', oob_score=True, bootstrap=True),
    search_spaces=param_space,
    n_iter=50,          
    cv=5,
    scoring='accuracy',           
    n_jobs=-1,            
    random_state=seed,
    verbose=0
)

In [ ]:
opt.fit(X_train_full_scaled, y_train_full)

In [ ]:
best_model = opt.best_estimator_

for score in best_model.feature_importances_:
    print(score)

In [ ]:
y_pred_best = best_model.predict(X_test_scaled)
print("Model Accuracy:", accuracy_score(y_test, y_pred_best))

In [ ]:
cm = confusion_matrix(y_test, y_pred_best, labels=best_model.classes_)

print("Confusion Matrix:")
print(cm)

## Weighted Approach - Weighted Predictions

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=seed)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_val_scaled = scaler.transform(X_val)

In [ ]:
opt.fit(X_train_scaled, y_train)

In [ ]:
class_indices = {label: idx for idx, label in enumerate(best_model.classes_)}

tree_accuracies = [tree.score(X_val_scaled, y_val) for tree in best_model.estimators_]
tree_weights = np.array(tree_accuracies)
tree_weights /= tree_weights.sum()

tree_preds = np.array([tree.predict(X_test_scaled) for tree in best_model.estimators_])

weighted_votes = np.zeros((X_test_scaled.shape[0], len(best_model.classes_)))

for i, preds in enumerate(tree_preds):
    for j, pred in enumerate(preds):
        class_index = class_indices[pred]
        weighted_votes[j, class_index] += tree_weights[i]

final_preds = np.argmax(weighted_votes, axis=1)


accuracy = accuracy_score(y_test, final_preds)
print(f"Final weighted prediction accuracy: {accuracy:.4f}")

print("\nClassification Report for Tuned Model:\n", classification_report(
    y_test, final_preds, target_names=[str(cls) for cls in label_encoder.classes_]
))

cm = confusion_matrix(y_test, final_preds, labels=best_model.classes_)

print("Confusion Matrix:")
print(cm)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


## XGBoost

In [ ]:
from xgboost import XGBClassifier
xgmodel = XGBClassifier(use_label_encoder=False, eval_metric='logloss')


In [ ]:
xgmodel.fit(X_train_full_scaled, y_train_full)
y_pred = xgmodel.predict(X_test_scaled)

accuracy_score(y_test, y_pred)